# Cross Validation, Model Selection & Regularisation

Signification des émoticones :
- 🌞 : documentations importantes
- 👀 : documentations intéressantes à connaître
- 🌚 : en complément
- (vide) : à vous de voir

In this notebook we will introduce the concepts below, and how they can be implemented in `scikit-learn`.
* Cross validation - a method for estimating the test error rate when test data is not available
* Model selection - how we use cross validation to select which model (from a selection) we should use for a particular data set
* Regularisation - an adaptation of linear regression to make it more flexible

[This video](https://www.youtube.com/watch?v=DQWI1kvmwRg) describes some of the ideas you will face in the coming notebook. The ideas we are covering here are described much more throughly in **ISLR** (see suggested sections in module overview).

🌚 https://trevorhastie.github.io/ISLR/

In [ ]:
import pandas as pd
import numpy as np

## Cross validation

We have already seen the notion of splitting *test* and *training sets* in order to asses model performance. Now we will introduce the idea of *validation sets*.

Once again **ISTL** (Section 5.1) provides a very clear overview of how these techniques work:

>Resampling methods are an indispensable tool in modern statistics. They
involve repeatedly drawing samples from a training set and refitting a model
of interest on each sample in order to obtain additional information about
the fitted model. For example, in order to estimate the variability of a linear
regression fit, we can repeatedly draw different samples from the training
data, fit a linear regression to each new sample, and then examine the
extent to which the resulting fits differ. Such an approach may allow us to
obtain information that would not be available from fitting the model only
once using the original training sample.

>Resampling approaches can be computationally expensive, because they
involve fitting the same statistical method multiple times using different
subsets of the training data. However, due to recent advances in computing
power, the computational requirements of resampling methods generally
are not prohibitive. [...] cross-validation can be used to estimate the test
error associated with a given statistical learning method in order to evaluate
its performance, or to select the appropriate level of flexibility. The process
of evaluating a model’s performance is known as model assessment, whereas model
the process of selecting the proper level of flexibility for a model is known as assessment
model selection.

The diagram below illustrates this process. Here the train and validation sets are used to do model assesment and model selection (NOTE: the test set is strictly forbidden from being used in any way during this process!). Once a model is selected the test set is used to do a final assesment of performance to see if the model selected will generalise as well as predicted.

<img src="../Images/testtrainvalid.png" width="450px">

There is a simple overview these ideas 👀[here](https://towardsdatascience.com/training-vs-testing-vs-validation-sets-a44bed52a0e1/), and a more thorough overview 🌚[here](https://machinelearningmastery.com/difference-test-validation-datasets/).

🌞 This video of Machine learnia also explains the concept of cross validation very well : https://www.youtube.com/watch?v=w_bLGK4Pteo&ab_channel=MachineLearnia

### K-fold cross validation

A very common method called *k-fold* is often used, which actually splits the training set multiple times. This allows us to assess the accuracy of the model over $k$ validation splits of data. The 🌚[image below](http://ethen8181.github.io/machine-learning/model_selection/model_selection.html) illustrates how this works for $k = 5$ splits.

<img src="../Images/kfold.png" width="450px">

The `scikit-learn` documentation offers a simple 🌞[example](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) of implementing k-fold on some dummy data. We will examine this below. NOTE: The `scikit-learn` documentation is **FANSTASTIC(!)** and contains working examples of every function, it should always be the first place you look when you wish to implement a new function.

**Task 1:** 

* Have a look at the code below and check you understand what is going on. (Add some print statements in various places to help.)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
X = np.array([['A', 'B'], ['C', 'D'], ['E', 'F'], ['G', 'H']])
y = np.array([1, 2, 3, 4])
kf = KFold(n_splits=3) # here we choose the number of folds (or splits) we will make
kf.get_n_splits(X)

for train_index, valid_index in kf.split(X): # kf.split(X) is an iterable which gives us the indices of the data in each fold
    X_train, X_valid = X[train_index], X[valid_index]
    y_train, y_valid = y[train_index], y[valid_index]

We can apply this to any data set to perform operations as it gives us the dataframe/numpy array indicies in each loop to select the appropriate data for each fold. For example we can apply this to the auto data set. This contains rows which correspond to individual cars and their attributes.

In [ ]:
auto = pd.read_csv('../Data/Auto.csv')
auto = auto[auto.horsepower != '?']
auto['horsepower'] = auto.horsepower.astype(int)
auto.reset_index(inplace=True, drop=True)
auto.head()

In [ ]:
mpg = auto.pop('mpg') # mpg will be our target and so we remove this into a seperate array

Say we are trying to predict 'mpg' from our other automobile data features. We can use KFold to iterate over the number of splits we choose.

**Task 2:**
* try adding print statements for the size of the dataframes in each split
* try increasing the number of splits and re-run your code
* use the code below to print a car name contained in the train and validation data set, for each split



In [ ]:
kf = KFold(n_splits=4) # here we choose the number of folds (or splits) we will make
kf.get_n_splits(auto)

print(kf)


split_counter = 1
for train_index, valid_index in kf.split(auto): # kf.split(X) is an iterable which gives us the indices of the data in each fold
    print('-'*60)
    print('This is split no: {}'.format(split_counter))
    split_counter += 1 
    X_train, X_valid = auto.iloc[train_index], auto.iloc[valid_index] # must use .iloc because its a dataframe this time
    y_train, y_valid = mpg[train_index], mpg[valid_index]
    
    # your code here
    print('Train car',X_train['name'].iloc[0])
    print('Validation',X_valid['name'].iloc[0])
    print('Real size', len(auto))
    print('Train size', len(X_train))
    print('Validation size', len(X_valid))

In [ ]:
#### Your solution here


In [ ]:
auto = auto.drop('name', axis=1, errors='ignore') # we do not need the car names so remove for now.

We can use the same loop to fit and evaluate our linear regression model on each train/validation split:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


split_counter = 1
mse_scores = [] # create empty list to append mse scores for each split
kf = KFold(n_splits=5)
kf.get_n_splits(auto)

for train_index, valid_index in kf.split(auto): # kf.split(X) is an iterable which gives us the indices of the data in each fold
    print('-'*60)
    print('This is split no: {}'.format(split_counter))
    print('Model results')
    split_counter += 1 
    X_train, X_valid = auto.iloc[train_index], auto.iloc[valid_index] # must use .iloc because its a dataframe this time
    y_train, y_valid = mpg[train_index], mpg[valid_index]
    
    
    #### fit polynomial to train data in this split
    lin_reg = LinearRegression()
    lin_reg.fit(X_train, y_train)
    
    #### eval & print MSE training results in this split
    mpg_train_pred = lin_reg.predict(X_train)
    mse_train = mean_squared_error(y_train, mpg_train_pred)
    print('training MSE: {0}'.format(mse_train))
    
    #### do the same for validation split
    mpg_valid_pred = lin_reg.predict(X_valid)
    mse_valid = mean_squared_error(y_valid, mpg_valid_pred)
    print('validation MSE: {0}'.format(mse_valid))
    
    mse_scores.append(mse_valid) # assign validation MSE score to list

When using k-fold cross validation we can analyse the validation MSE result for each split to assess the overall performance.

In [ ]:
mse_scores  = np.array(mse_scores)
print('VALIDATION SET MSE SCORES')
print('mean MSE:', mse_scores.mean())
print('std MSE:', mse_scores.std())

#### Cross validation in practice in sklearn
Most of the time we do not care about having access to each split. `Scikit-Learn` provide a much easier way to do all of this with the function `cross_val_score`. This allows us to do the same as above but in much less code.

In [ ]:
from sklearn.model_selection import cross_val_score
lin_reg = LinearRegression()

cv_scores = cross_val_score(lin_reg, auto, mpg, cv = 5, scoring='neg_mean_squared_error') 

print('mean MSE:',np.mean(-cv_scores))
print('std MSE:',np.std(-cv_scores))

In [ ]:
cv_scores

**Task 3:**

* Make sure you understand the output of the cross_val_score above (i.e. What is cv_scores?)

It returns one score per fold, which means one score per iteration. Scores of validation
* Why is the scoring defined as negative MSE? Do some research
Since it is a score, scikit-learn has a criterium to evaluate the models: the hights the score the better the model, but  since MSE works in the opposite way, they use the negative value: in MSE 5 is better than 15 so but 15 is bigger si we take the negative values: -5 is bigger than -15 so is consistent. The better the score the better the model

* Investigate what the `cross_val_predict` function does.

It returns the prediction or each observation when they are in validation, so when they havent been used for training.

* Import and implement `cross_val_predict`on the same data as above.
* What are the outputs of this function?

Prediction of each 'input' when they were in validation group, so as data the model has never seen
* Can you use these to evaluate the results of your cross validation?

Yes, we can compare the prediction to the real values, for example, calculating the MSE
* Do the cross-validation scores give you confidence this model is providing a useful prediction?

We cannot determine whether the model provides useful predictions based only on these values. We should also compare its performance with a baseline model.

In [ ]:
# solution
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_squared_error


predictions = cross_val_predict(lin_reg, auto, mpg, cv = 5)
mean_squared_error(mpg, predictions) #its very similar to the result given by cross_val_scores

## Exercise 1: wine cross-validation

You must predict the alcohol content of various wines based on their other attributes.

* Split the data into train and test data sets (Ensure you use the option: `random_state = 28`).
* Perform linear regression using k-fold cross validation(ensure you use 5 folds). Return the cross validation MSE errors. Return the mean and standard deviations of these.
* Evaluate the performance of the model on the test set.
* Compare the cross-validation error and the test error (MSE). What do you find?

The cross_validation and test MSE are almost identical. This suggests that the model performs consistently on different subsets of data and that its predictions generalise well to unseen observations.

* Try removing the random_state option. What happens to your results? Explain why.

The result vary slightly. This is because it depends on which observations we are using for each group (train-test split is different/random). It is one of the reaosns why we use k-fold cross validation. 

In [ ]:
#### your solution here
from sklearn.model_selection import train_test_split
wine = pd.read_csv('../Data/wine.csv')
X = wine.drop(columns = 'Alcohol')
y = wine.pop('Alcohol')

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=28)
lin_reg = LinearRegression()

cv_scores = cross_val_score(lin_reg, X_train, y_train, cv = 5, scoring='neg_mean_squared_error') 
print('mean MSE:',np.mean(-cv_scores))
print('std MSE:',np.std(-cv_scores))

##Entrenamos el modelo con train y lo testeamos con test
model = LinearRegression()
model.fit(X_train, y_train)
y_pred_test = model.predict(X_test)
mse_test = mean_squared_error(y_test, y_pred_test)
print('test MSE: {0}'.format(mse_test))

## Regularisation

An alternative to choosing models which contains smaller numbers of features is to use a method that *constrains* or *regularises* the coefficent estimates assigned to each feature, or that shrinks the coefficient towards zero. This technique is very similar to *least squares* which we have been using until now. Please refer to Section in 6.2 **ISTL** for a fuller explanation of this.

When we move to use a regularised linear regression for prediction, the additional term means that we now have a model parameter that requires setting or tuning. These terms are referred to as *hyperparameters* in machine learning. In practice this introduces another additional unknown parameter which we must choose somewhere in our modelling. It is common practice to run several models, each with different values of this hyperparameter, and then assess the error of each using cross validation for comparison.

**Lasso vs Ridge** (just for information curiosity, no need to go in depth or spend to much time to understand the maths)

For now we will focus on how to implement Lasso and Ridge regression in sklearn. These are both types of regularised linear regression.

Lasso: objectif is to minimize
$$ RSS + \lambda \sum |\beta | $$
* can force coefficients exactly to zero: behaves thus as variable selection


Ridge: objectif is to minimize
$$ RSS + \lambda \sum \beta_i^2 $$
* does not force coefficients exactly to zero
* interestig when there are more predictors than observations

see further reading ILS, search for Ridge and Lasso

#### Lasso regression in sklearn

In this example we aim to predict credit rating of individual customers. To train and predict using a Lasso regression we follow much the same procedure as we have seen before in `scikit-learn`.

Additional sources for further reading :

👀 https://www.datacamp.com/community/tutorials/tutorial-ridge-lasso-elastic-net

🌚 https://eric.univ-lyon2.fr/ricco/cours/slides/regularized_regression.pdf

In [ ]:
credit = pd.read_csv('../Data/credit_modified.csv')
rating = credit.pop('Rating')
credit.head()

In [ ]:
#### splitting train and test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(credit, rating, random_state = 91)

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha = 0.1)
cv_scores = cross_val_score(lasso, X_train, y_train, cv = 5, scoring='neg_mean_squared_error')
print('mean MSE:',np.mean(-cv_scores))
print('std MSE:',np.std(-cv_scores))

Here we can alter the alpha parameter to change the amount of regularisation the model uses (try this yourself! - vary the value by at least factors of 10). With increases in regularisation we expect a reduction in the *variance* of the model.

#### Using Lasso with grid search

In practice we do not want to vary hyperparameters by hand to find which value is best (the model with minimum cross validation error). Of course `scikit-learn` has a function that automates this for you. Using `GridSearchCV` we pass a dictionary of parameter values we wish to investigate. The function will fit each model we have listed and calculate the cross validation error of each. It provides all the results through the object it returns.

In [ ]:
from sklearn.model_selection import GridSearchCV
lasso = Lasso(max_iter=10000)

param_grid = [
 {'alpha': [0.001, 0.01, 0.1, 1, 3, 10, 100, 1000]}
 ]

grid_search = GridSearchCV(lasso, param_grid, cv=10, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

grid_results = pd.DataFrame(grid_search.cv_results_)

grid_results

`GridSearchCV` also returns a model (with the best hyperparmeter combination it found) which has been fitted one final time to all of the training data. Therefore it is ready to make predictions on the testing set. The model can be accessed like this:

In [ ]:
grid_search.best_estimator_

**Task 4:**
* How many times will the lasso model be fitted when the GridSearchCV function is called above?

We have 8 different values for alpha and 10 folds, so its 80 trainings

<details><summary>Hint</summary><br>
Check what the `refit=True` parameter does in GridSearchCV
</details>

* Look through the columns of the `grid_results` dataframe. Try and understand what the table contains.

👀 https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

## Model selection

Congratulations! You have just done your first model hyperparameter tuning in `scikit-learn`! 

If we have a dataset for which we are interested in developing a predictive model. We do not know beforehand which model will perform best for this particular data or problem. Therefore, we fit and evaluate a number of different models to our data. The models could also be of varying type as well as flexibility (e.g. random forests, support vector machines, linear regression). We then need to decide which of our models we will choose to use in our final product.

As **ISLR** states:
> "we can directly estimate the test error using the validation set and cross-validation methods
discussed in Chapter 5. We can compute the validation set error or the
cross-validation error for each model under consideration, and then select
the model for which the resulting estimated test error is smallest."

This works as a simple rule, which we will follow for the remainder of this notebook. However in practice the selection can sometimes be a bit more nuanced. Read more detail 🌚[here](https://machinelearningmastery.com/a-gentle-introduction-to-model-selection-for-machine-learning/).

#### Task 5: Ridge regression in sklearn

Another type of regularised linear model is know as *Ridge regression*.

* Repeat the model prediction process above on the credit data but use a ridge regression model.
* Try replacing `GridSearchCV` with `RandomizedSearchCV`
* How do these functions differ?

GridSearchCV tests all possible combinations of hyperparameters, whereas RandomSearchCV test a fixed number of randomly selected combinatios (n_iter)

In [ ]:
#### Your solution here

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
ridge = Ridge(max_iter=10000)

param_grid = [
 {'alpha': [0.001, 0.01, 0.1, 1, 3, 10, 100, 1000]}
 ]

grid_search = GridSearchCV(ridge, param_grid, cv=10, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

grid_results = pd.DataFrame(grid_search.cv_results_)

grid_results

In [ ]:
#### change to randomised search CV
#### Your solution here

from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import Ridge
ridge = Ridge(max_iter=10000)

param_grid = [
 {'alpha': [0.001, 0.01, 0.1, 1, 3, 10, 100, 1000]}
 ]

random_search = RandomizedSearchCV(ridge, param_grid, cv=10, n_iter = 5, scoring='neg_mean_squared_error')
random_search.fit(X_train, y_train)

random_results = pd.DataFrame(random_search.cv_results_)

random_results


## Exercise 2: Moneyball

Moneyball, as well as being a fantastic story, is also a true story of statistical methods being applied in a real world context to make predictions for decision making. [The film Moneyball](https://www.youtube.com/watch?v=-4QPVo0UIzc) is well worth a watch if you have time. As well as in baseball most major competitive sports teams are now using data science to improve their performance, e.g. [football](https://www.isportconnect.com/the-rise-of-data-driven-football-how-ai-and-analytics-are-shaping-the-future-of-the-game/),...

In this excercise you have been hired by Oakland Athletics general manager Billy Beane. Your first mission is to predict the salary each player will make based on other information that is available. This will allow Billy to understand what price he should pay for players in the next transfer season.

You must:
* Import and prepare the data
* Create a train and test set
* Implement a regularised model of your choice (Ridge or Lasso)
* Choose optimal parameters for your regularised model
* Estimate test-error using k-fold cross validation
* Calculate the true test-error
* Run a base line model to compare your model results. Base line model is the most simple approach based on strategy of choice (mean or other). It is then used as reference to conclude whether more complex models are better or not: see DummyRegressor in the sklearn library.

HINT 1
* Some values are missing. You can drop these rows.

HINT 2
* Some columns do not contain numerical values. You can drop these columns. In the Machine Learning model you'll learn more about labeling categorical data

In [ ]:
#### Your solution here
# import the hitters.csv dataset


In [ ]:
# clean data



make train test split

In [ ]:
# make train test split


linear regression model build using CV to estimate what our model performance will be on data it hasnt seen

In [ ]:
##### using k-fold cross val score


In [ ]:
# regularised model - lasso build


In [ ]:
# Print the best hyperparameter for model performance....


**One important thing** to realise is that when we run our `GridSearchCV` for our lasso model we are changing the flexibility of our model and estimating our overall test error. In essence we are estimating this curve below, which we saw in the text book!

<img src="../Images/bias_var_tradeoff.png" width="300px">

We can plot our cv values for each of our model flexibilities (alpha values)....

In [ ]:
# plot cv values against alpha parameters

**Task 6:**
* Ridge regression works best when the input variables are standardised. (See section 6.2 **ISLR** for more details.). Try standardising your data before running your model. Do you find different results?
* does this model outperform a simple linear regression model?
* Which variables are most important in the model?
* Make a pairplot or scatter matrix: pd.plotting.scatter_matrix

**These and other things to consider in this problem...**

* We must actually standardise our *X* variables by scaling each *X* by the standard deviation, if not Ridge will not penalise each variable in the same way. We will learn a robust and easy way to do this in the intro to ML module.
* Once our variables are scaled we can use the $\beta$ coefficients that our model finds to see which has a greater influence in our model. We can find as we see below.
* The reason our model does not perform so well is that most of our *X* variables do not have a very clear linear relationship with *y* (see below)! Using non-linear models could be of use in this problem...again we will cover these soon!

In [ ]:
# Your solution here

#standardize data


In [ ]:
# run model and compare with results from non-standardized X


In [ ]:
#### solution plot scatter matrix


## To summarize

* **K-fold cross-validation** splits training data into k subsets, using each as a validation set once. This process estimates model performance and helps select optimal hyperparameters. Finally, the chosen model is evaluated on a separate, held-out test set to assess its generalization ability.

* **cross_val_score** simplifies evaluation by computing metrics (e.g., MSE) across folds, while cross_val_predict generates predictions for each sample.

* **Lasso (L1)** and **Ridge (L2)** regression constrain coefficients to prevent overfitting:
> * Lasso can zero out irrelevant features (feature selection).
> * Ridge shrinks coefficients but rarely to zero, useful when predictors outnumber observations.

* Hyperparameter tuning: **GridSearchCV** automates searching for the optimal regularization strength (alpha) via cross-validation.

* **RandomizedSearchCV** efficiently explores hyperparameters by sampling randomly, unlike exhaustive GridSearchCV.